In [ ]:
# --- 추론 함수 정의 ---
def get_prompt_message(row, image_dir):
    """
    4장의 프레임과 문장을 조합하여 Qwen2-VL-2B-Instruct 모델에 보낼 프롬프트 메시지를 구성합니다.
    """
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({
            "type": "image",
            "image": img_path,
        })
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]
    return messages

In [ ]:
from qwen_vl_utils import process_vision_info

row = test_df.iloc[0]

messages = get_prompt_message(row, TEST_IMAGE_DIR)

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

inputs = inputs.to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False
    )

generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print("모델 원본 출력:", output_text)
print("제출 형식 변환:", parse_model_output(output_text))

In [ ]:
import ast
import re

def parse_model_output(output_text):
    """
    모델 출력에서 [1, 2, 3, 4] 형태의 순열을 찾고
    대회 Answer 형식으로 변환합니다.

    파싱에 실패하면 [1, 2, 3, 4]가 아니라 None을 반환합니다.
    """
    pattern = r"\[\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*\]"
    match = re.search(pattern, output_text)

    if match is None:
        return None

    try:
        result = ast.literal_eval(match.group())

        if (
            isinstance(result, list)
            and len(result) == 4
            and sorted(result) == [1, 2, 3, 4]
        ):
            # 모델 출력:
            # 시간순으로 나열된 Input 번호
            #
            # 대회 Answer:
            # 각 Input이 실제 몇 번째 프레임인지
            submission_answer = [0] * 4

            for chronological_position, image_num in enumerate(result, start=1):
                submission_answer[image_num - 1] = chronological_position

            return submission_answer

    except (ValueError, SyntaxError, TypeError):
        return None

    return None

In [ ]:
from tqdm.auto import tqdm
predictions = []

print("Starting inference...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    messages = get_prompt_message(row, TEST_IMAGE_DIR)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    pred_list = parse_model_output(output_text)

    predictions.append({
        "Id": row["Id"],
        "Answer": str(pred_list)
    })

In [ ]:
submit_path = "/content/outputs/submission.csv"

submission_df = pd.DataFrame(predictions)
submission_df.to_csv(submit_path, index=False)

print("저장 완료:", submit_path)
display(submission_df.head())